In [2]:
# firebase-admin 설치
!pip install firebase-admin

     ---------------------------------------- 0.0/13.0 MB ? eta -:--:--
     ---------- ----------------------------- 3.4/13.0 MB 22.3 MB/s eta 0:00:01
     ------------- -------------------------- 4.5/13.0 MB 11.7 MB/s eta 0:00:01
     ------------------ --------------------- 6.0/13.0 MB 10.0 MB/s eta 0:00:01
     ------------------------- -------------- 8.4/13.0 MB 10.4 MB/s eta 0:00:01
     ------------------------------- -------- 10.2/13.0 MB 9.8 MB/s eta 0:00:01
     ----------------------------------- ---- 11.5/13.0 MB 9.5 MB/s eta 0:00:01
     ---------------------------------------- 13.0/13.0 MB 9.6 MB/s eta 0:00:00
  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/3.4 MB ? eta -:--:--
   ---------------------------------------- 3.4/3.4 MB 33.3 MB/s eta 0:00:00
   ---------------------------------------- 0.0/4.2 MB ? eta -:--:--
   ---------------------------------------- 4.2/4.2

In [3]:
# 인증 키 업로드
from google.colab import files
uploaded = files.upload()  # 비밀키 JSON 파일 선택해서 업로드 (예: deepsentry-xxx.json)

ModuleNotFoundError: No module named 'google.colab'

In [ ]:
# Firebase Admin 초기화 및 Firestore 클라이언트 생성
import firebase_admin
from firebase_admin import credentials, firestore

import os

# 업로드한 파일명(파일명 수정하지 말고 인증키 업로드만)
cred = credentials.Certificate("deepsentry-39f58-firebase-adminsdk-fbsvc-645bb22a0a.json")
firebase_admin.initialize_app(cred)

db = firestore.client()

# 여기부터 실행 

In [ ]:
# --- Imports ---
import cv2
import os
import time  # 시간 관리를 위해 필요
import datetime # 업로드 시간 기록 및 포맷팅

# Firebase imports
import firebase_admin
from firebase_admin import credentials, firestore

# YOLO import (ultralytics 라이브러리 필요)
from ultralytics import YOLO 

# --- 상수 및 설정 (님의 환경에 맞게 수정하세요) ---
# 파일 경로 설정
# DATASET_BASE_DIR과 CUSTOM_RUNS_DIR는 모델 로드 시 필요할 수 있습니다.
# 모델 경로 및 클래스 경로를 실제 경로로 정확하게 지정하세요.
# model.names가 YAML 파일의 names와 일치한다면 data_cf는 필요 없을 수 있습니다.
# 여기서는 model.names를 직접 사용하는 것으로 가정합니다.
MODEL_PATH = r'C:\Users\AREU\Desktop\PolyProject\runs\train\weights\best.pt' # 학습된 모델 파일 경로
FIREBASE_CREDENTIAL_PATH = r'C:\Users\AREU\Desktop\PolyProject\databaseuploading\fkey\어쩌구저저구.json' # <--- 이 부분을 수정

# 웹캠 설정
WEBCAM_INDEX = 0 # 보통 0번이 기본 웹캠
WEBCAM_WIDTH = 640
WEBCAM_HEIGHT = 480

# 탐지 및 업로드 조건 설정
CONFIDENCE_THRESHOLD = 0.75             # 객체 신뢰도 임계값
CONTINUOUS_DETECTION_SECONDS = 3        # 연속 탐지 지속 시간 (초)
UPLOAD_COOLDOWN_SECONDS = 5 * 60        # 같은 클래스 재업로드 대기 시간 (초, 5분)

# Firebase에 기록될 고정 정보
LOGIN_ID = "Admin"
DET_LOCATION = "Cam1"

# 탐지 대상 클래스 목록 (님의 YAML names와 일치해야 함)
TARGET_CLASSES = ['bear', 'boar', 'waterdeer', 'human'] # 사람 클래스 추가 시 'person'도 추가해야 함

# --- Firebase Admin 초기화 및 Firestore 클라이언트 생성 (스크립트 시작 시 한 번만) ---
try:
    # 이미 초기화되었는지 확인 (Colab 등 환경에서 셀을 여러 번 실행할 경우 대비)
    if not firebase_admin._apps:
        cred = credentials.Certificate(FIREBASE_CREDENTIAL_PATH)
        firebase_admin.initialize_app(cred)
        print("✅ Firebase Admin initialized.")
    else:
         print("✅ Firebase Admin already initialized.")

    db = firestore.client()
    print("✅ Firestore client created.")

except Exception as e:
    print(f"❌ Firebase initialization failed: {e}")
    print("Firebase 연결 오류로 프로그램을 종료합니다.")
    exit() # Firebase 연결 실패 시 프로그램 종료


# --- YOLO 모델 로딩 (스크립트 시작 시 한 번만) ---
try:
    model = YOLO(MODEL_PATH)
    print(f"✅ YOLO model loaded from {MODEL_PATH}")
    # 모델이 인식하는 클래스 목록과 TARGET_CLASSES 일치 확인 (선택 사항, 디버깅에 유용)
    print(f"   Model recognizes classes: {model.names}")
    # TARGET_CLASSES 리스트에 있는 이름이 model.names 리스트에도 있어야 합니다.
    if not all(cls_name in model.names for cls_name in TARGET_CLASSES):
         print("⚠️ WARNING: TARGET_CLASSES에 모델이 인식하지 못하는 클래스 이름이 포함되어 있을 수 있습니다.")

except Exception as e:
    print(f"❌ Failed to load YOLO model from {MODEL_PATH}: {e}")
    print("모델 로드 오류로 프로그램을 종료합니다.")
    exit() # 모델 로드 실패 시 프로그램 종료


# --- 웹캠 초기화 (스크립트 시작 시 한 번만) ---
cap = cv2.VideoCapture(WEBCAM_INDEX)
if not cap.isOpened():
    print(f"❌ Failed to open webcam with index {WEBCAM_INDEX}.")
    print("웹캠 초기화 오류로 프로그램을 종료합니다.")
    exit() # 웹캠 초기화 실패 시 프로그램 종료

cap.set(cv2.CAP_PROP_FRAME_WIDTH, WEBCAM_WIDTH)
cap.set(cv2.CAP_PROP_FRAME_HEIGHT, WEBCAM_HEIGHT)
print(f"✅ Webcam opened with resolution {WEBCAM_WIDTH}x{WEBCAM_HEIGHT}")


# --- 탐지 상태 및 업로드 쿨다운 관리를 위한 변수 ---
# 각 클래스별 연속 탐지 시작 시간을 저장 {'class_name': start_timestamp}
continuous_detection_start_time = {}

# 각 클래스별 마지막 업로드 시간을 저장 {'class_name': last_upload_timestamp}
# 초기값은 과거 시간으로 설정하여 프로그램 시작 후 바로 업로드 가능하게 함
last_upload_time = {cls_name: 0 for cls_name in TARGET_CLASSES}


print("\n▶️ Detection and Upload Process Started.")
print(f"   Continuous detection threshold: {CONTINUOUS_DETECTION_SECONDS} seconds")
print(f"   Upload cooldown period: {UPLOAD_COOLDOWN_SECONDS} seconds ({UPLOAD_COOLDOWN_SECONDS/60} minutes)")
print("   Press 'q' key on the webcam window to exit.")
print("-" * 50)

# --- 메인 루프: 웹캠 프레임 처리 ---
while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        print("Error: Can't receive frame (stream end?). Exiting ...")
        break

    # 현재 프레임에서 탐지된 대상 클래스 목록을 저장할 집합
    current_frame_detected_target_classes = set()

    # --- 객체 탐지 및 결과 처리 ---
    # verbose=False로 설정하면 추론 과정의 상세 로그 출력을 줄여 터미널을 깔끔하게 유지할 수 있습니다.
    results = model(frame, verbose=False)

    for result in results:
        boxes = result.boxes # 탐지된 객체의 bounding box 정보

        for box in boxes:
            conf = box.conf.item()        # 신뢰도
            cls_index = int(box.cls.item()) # 클래스 인덱스

            # 모델의 클래스 목록 범위를 벗어나지 않는지 확인
            if cls_index < len(model.names):
                cls_name = model.names[cls_index] # 클래스 이름 가져오기

                # 설정된 신뢰도 임계값과 대상 클래스 목록에 해당하는지 확인
                if conf >= CONFIDENCE_THRESHOLD and cls_name in TARGET_CLASSES:
                    # 이 객체는 유효하게 탐지된 대상 클래스입니다.

                    # 현재 프레임 탐지 목록에 추가
                    current_frame_detected_target_classes.add(cls_name)

                    # --- (Optional) 바운딩 박스 및 라벨 그리기 ---
                    # 탐지된 객체에 대해 프레임에 그립니다.
                    x1, y1, x2, y2 = map(int, box.xyxy[0])
                    label = f"{cls_name} {conf:.2f}"
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2) # 초록색 박스
                    cv2.putText(frame, label, (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)


    # --- 연속 탐지 상태 업데이트 및 쿨다운/업로드 로직 ---
    current_time = time.time() # 현재 시간 (타임스탬프)

    # 이전 프레임에서 연속 탐지 중이었지만 현재 프레임에서 탐지되지 않은 클래스 찾기
    # -> 연속성 깨짐 -> 타이머 초기화
    classes_to_reset = [
        cls_name for cls_name in continuous_detection_start_time
        if cls_name not in current_frame_detected_target_classes
    ]
    for cls_name in classes_to_reset:
        # print(f"💡 연속 탐지 끊김: {cls_name}. 타이머 초기화.")
        del continuous_detection_start_time[cls_name] # 해당 클래스의 시작 시간 기록 삭제


    # 현재 프레임에서 탐지된 대상 클래스들에 대해 연속 탐지 시작 시간 기록 또는 업데이트
    for cls_name in current_frame_detected_target_classes:
         # 이 클래스가 연속 탐지 기록에 없다면 (새로운 연속 탐지 시작)
         if cls_name not in continuous_detection_start_time:
              # 현재 시간을 연속 탐지 시작 시간으로 기록
              continuous_detection_start_time[cls_name] = current_time


    # 연속 탐지 조건을 만족하고 쿨다운 기간이 지난 클래스 확인 및 업로드
    # 현재 continuous_detection_start_time에 남아있는 클래스들만 대상
    # dictionary를 순회하면서 항목을 삭제할 수 있으므로 list로 복사하여 순회
    for cls_name, start_ts in list(continuous_detection_start_time.items()):
        # 1. 3초 연속 탐지 조건 확인
        if current_time - start_ts >= CONTINUOUS_DETECTION_SECONDS:
            # print(f"⏰ {cls_name} {CONTINUOUS_DETECTION_SECONDS}초 이상 연속 탐지됨.")

            # 2. 5분 쿨다운 조건 확인
            last_upload_ts = last_upload_time.get(cls_name, 0) # 마지막 업로드 시간 가져오기 (없으면 0)

            if current_time - last_upload_ts >= UPLOAD_COOLDOWN_SECONDS:
                print(f"\n⬆️ 조건 만족: {cls_name} 탐지 로그 업로드 시작. (마지막 업로드: {datetime.datetime.fromtimestamp(last_upload_ts).strftime('%Y-%m-%d %H:%M:%S')})")

                # --- Firebase에 데이터 업로드 ---
                try:
                     doc_ref = db.collection("LogRecord").document()  # Firestore 자동 ID 문서 생성
                     det_date = datetime.datetime.now().strftime("%Y-%m-%d") # 실제 탐지 날짜
                     det_time_str = datetime.datetime.now().strftime("%H:%M:%S") # 실제 탐지 시간

                     doc_ref.set({
                         "LoginID": LOGIN_ID,         # 고정 사용자 ID
                         "DetDate": det_date,
                         "DetTime": det_time_str,
                         "DetAnimal": cls_name,       # 실제 탐지된 클래스 이름
                         "DetLocation": DET_LOCATION  # 고정 위치 정보
                     })
                     print(f"✅ Firestore에 탐지 로그 성공적으로 업로드: {cls_name}")

                     # --- 마지막 업로드 시간 업데이트 ---
                     last_upload_time[cls_name] = current_time # 해당 클래스의 마지막 업로드 시간을 현재 시간으로 업데이트

                     # 참고: 업로드 후에도 continuous_detection_start_time은 유지됩니다.
                     # 객체가 계속 탐지되면 3초 조건은 계속 만족하며, 다음 업로드는 쿨다운 이후에 가능해집니다.

                except Exception as e:
                    print(f"❌ Firestore 업로드 실패: {e}")

            # else:
                 # print(f"⏳ 쿨다운 중: {cls_name} 탐지되었으나 쿨다운 기간 내.")


    # --- 결과 프레임 출력 ---
    cv2.imshow('YOLO Webcam Detection', frame)

    # --- 종료 조건 확인 ---
    # 'q' 키를 누르면 루프 종료
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# --- 자원 해제 ---
cap.release()
cv2.destroyAllWindows()
print("-" * 50)
print("▶️ Program terminated.")

❌ Firebase initialization failed: [Errno 2] No such file or directory: 'C:\\Users\\AREU\\Desktop\\PolyProject\\databaseuploading\\fkey\\어쩌구저저구.json'
Firebase 연결 오류로 프로그램을 종료합니다.
✅ YOLO model loaded from C:\Users\AREU\Desktop\PolyProject\runs\train\weights\best.pt
   Model recognizes classes: {0: 'bear', 1: 'boar', 2: 'waterdeer', 3: 'human'}
⚠️ WARNING: TARGET_CLASSES에 모델이 인식하지 못하는 클래스 이름이 포함되어 있을 수 있습니다.
❌ Failed to open webcam with index 0.
웹캠 초기화 오류로 프로그램을 종료합니다.
✅ Webcam opened with resolution 640x480

▶️ Detection and Upload Process Started.
   Continuous detection threshold: 3 seconds
   Upload cooldown period: 300 seconds (5.0 minutes)
   Press 'q' key on the webcam window to exit.
--------------------------------------------------
--------------------------------------------------
▶️ Program terminated.


: 